# Hasnain, Han & Economo (2025) on DANDI:000363: reading G2 of 'good'

A second notebook for the same entry. It repeats the tests of the Figure 4 claims on the ALM and
medulla sessions under reading G2 of the unit selection, where the main notebook
(`reproduce_hasnain_2025.ipynb`) uses reading G1. The paper selects "sessions with at least 30
units classified as 'good' in each region of interest" (Methods, "Curation of Mesoscale Activity
Map Dataset"). The units table has two 'good' labels: `classification` (G1) and `unit_quality`
(G2). Only units classified good carry a CCF label, so under G2 a unit takes the label of the
nearest unit classified good on the same probe, by the CCF coordinates of their electrodes, within
20 um (`map_subspace.transfer_labels`; an assumption). The main notebook gives the session and unit
counts under both readings and the check of the label transfer.

Dataset: DANDI:000363, version 0.231012.2129, streamed. Claims tested here, under G2:
fig4b-medulla-selectivity, fig4c-alm-null-potent-similar, fig4c-medulla-null-smaller,
fig4d-decoding-asymmetry and fig4e-medulla-null-lower-dimension. fig4d-alm-predicts-medulla-null
and fig4f-medulla-null-not-spanning-alm are tested in the main notebook only (the first needs the
bilateral sessions, 31 under G2; the second is left to the main
notebook to keep this one within the time limit). Methods, readings and seeds are those of the main notebook; the readings run here are the
main reading (W1, B1, D1, baseline z-score) and B2, on which fig4c-medulla-null-smaller rests in
the main notebook.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "2")
import collections
import resource
import time
import zlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import map_subspace as ms

T0 = time.time()
KEY = "hasnain-2025"


def seed(claim, k=0):
    return zlib.crc32(f"{KEY}/{claim}".encode()) + k


SEED_OFFSETS = [0, 1000, 2000, 3000, 4000]
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## Data check

Units tables of every file, the G2 selection, and for each selected session the trials kept, the
movement threshold and the clock checks of the main notebook (tongue latency after the go cue; lick
direction decoded from spike counts 0.8 to 1.0 s after the go cue against shuffled labels).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

assets = ms.asset_urls()
tables = ms.survey(assets)
urls = dict(assets)
bil_g2, alm_med = ms.select_sessions(tables, label="quality")
print("G2: bilateral sessions %d; ALM and medulla sessions %d from %d mice, %d ALM and %d medulla units"
      % (len(bil_g2), len(alm_med), len({p.split("/")[0] for p in alm_med}),
         sum(len(v["ALM"]) for v in alm_med.values()), sum(len(v["medulla"]) for v in alm_med.values())))
_, alm_med_g1 = ms.select_sessions(tables)
print("ALM and medulla sessions also selected under G1:", sum(p in alm_med_g1 for p in alm_med))

W = ms.WINDOWS["W1"]
sessions, rows = {}, []
labels_seen = collections.defaultdict(collections.Counter)
rng_check = np.random.default_rng(seed("map-session-selection", 1))
for p, regions in alm_med.items():
    ids = sorted(set(regions["ALM"]) | set(regions["medulla"]))
    s = ms.load_session(urls[p], ids)
    for k in ("outcome", "photostim_duration", "early_lick"):
        labels_seen[k].update(s["trials"][k])
    gos = ms.go_times(s)
    keep = ms.trial_mask(s)
    go = gos[keep]
    speed = ms.speed_bins(s, go, W)
    finite = speed[np.isfinite(speed) & (speed > 0)]
    log_thr = ms.otsu_threshold(np.log10(finite))
    f = ms.open_file(urls[p])
    tg = f["acquisition/BehavioralTimeSeries/Camera0_side_TongueTracking"]
    tts, tlik = tg["timestamps"][:], tg["data"][:, 2]
    hit = np.array(s["trials"]["outcome"])[keep] == "hit"
    lat = []
    for g in go[hit]:
        lo, hi = np.searchsorted(tts, [g, g + 1.0])
        k = np.where(tlik[lo:hi] > 0.9)[0]
        lat.append(tts[lo + k[0]] - g if len(k) else np.nan)
    right = np.array(s["trials"]["trial_instruction"])[keep] == "right"
    X = np.stack([np.diff(np.searchsorted(s["spikes"][i], np.c_[go + 0.8, go + 1.0]), axis=1)[:, 0] for i in ids], axis=1)
    clf = LogisticRegression(C=0.1, max_iter=2000)
    acc = cross_val_score(clf, X, right, cv=5).mean()
    acc_sh = cross_val_score(clf, X, rng_check.permutation(right), cv=5).mean()
    sessions[p] = dict(sess=s, keep=keep, go=go, speed=speed, log_thr=log_thr)
    rows.append(dict(session=p.split("/")[1][:30], G1=p in alm_med_g1, ALM=len(regions["ALM"]),
                     ALM_side="+".join(regions["alm_sides"]), medulla=len(regions["medulla"]),
                     trials=len(keep), kept=int(keep.sum()), hits=int(hit.sum()), speed_thr=round(10 ** log_thr, 1),
                     tongue_lat_ms=round(1000 * np.nanmedian(lat)), acc_go_08=round(acc, 2), acc_shuffled=round(acc_sh, 2)))
    del f
sel_df = pd.DataFrame(rows)
print(sel_df.to_string())
print("trial labels:", {k: dict(v) for k, v in labels_seen.items()})
print("decoding above shuffled labels in %d of %d sessions; tongue latency %d to %d ms"
      % ((sel_df.acc_go_08 > sel_df.acc_shuffled + 0.1).sum(), len(sel_df), sel_df.tongue_lat_ms.min(), sel_df.tongue_lat_ms.max()))
print("elapsed %.0f s, peak memory %.0f MB" % (time.time() - T0, resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e3))

## Claim fig4b-medulla-selectivity, reading G2

> Medullary reticular nucleus units are selective for lick direction in each epoch, most in the
> response epoch (sample 7%, delay 21%, response 49% of 724 units).

As in the main notebook (M7): correct trials, equal random samples of right and left trials,
AUROC per epoch, 1000 label shuffles, readings E1 and E2, Bonferroni (p < 0.01) and p < 0.05.

In [ ]:
def selectivity(rng):
    out = []
    for p, s in alm_med.items():
        v = sessions[p]
        trials = v["sess"]["trials"]
        out_arr = np.array(trials["outcome"])[v["keep"]]
        instr = np.array(trials["trial_instruction"])[v["keep"]]
        r_idx = np.where((out_arr == "hit") & (instr == "right"))[0]
        l_idx = np.where((out_arr == "hit") & (instr == "left"))[0]
        n = min(len(r_idx), len(l_idx))
        pick = np.r_[rng.choice(r_idx, n, replace=False), rng.choice(l_idx, n, replace=False)]
        is_right = np.r_[np.ones(n, bool), np.zeros(n, bool)]
        r = ms.rates([v["sess"]["spikes"][i] for i in s["medulla"]], v["go"][pick], ms.WINDOWS["W1"])
        tax = ms.WINDOWS["W1"][0] + ms.BIN * np.arange(r.shape[1])
        for j in range(r.shape[2]):
            row = dict(session=p.split("/")[1][:30], unit=s["medulla"][j], n_per_side=n)
            for ep, (a, b) in ms.EPOCHS.items():
                row[ep] = ms.selectivity_p(r[:, (tax >= a) & (tax < b), j].mean(1), is_right, rng)[1]
            out.append(row)
    return pd.DataFrame(out)


def summarize_sel(df, alpha):
    sig = {ep: df[ep] < alpha for ep in ms.EPOCHS}
    e1 = {"sample": sig["sample"], "delay": sig["early_delay"] | sig["late_delay"],
          "response": sig["early_response"] | sig["late_response"]}
    res = {f"E1 {k}": 100 * v.mean() for k, v in e1.items()}
    res.update({f"E2 {k}": 100 * v.mean() for k, v in sig.items()})
    return res


sel_runs = {k: selectivity(np.random.default_rng(seed("fig4b-medulla-selectivity", k))) for k in SEED_OFFSETS}
print("medulla units:", len(sel_runs[0]))
tab = pd.DataFrame({(off, a): summarize_sel(df, a) for off, df in sel_runs.items() for a in (0.01, 0.05)}).T
tab.index.names = ["seed offset", "alpha"]
print(tab.round(1).to_string())
print("expected: sample 7 +- 0.7, delay 21 +- 2.1, response 49 +- 4.9")
print("by session (claim seed, Bonferroni, E1):")
print(sel_runs[0].groupby("session").apply(lambda d: pd.Series(summarize_sel(d, 0.01))).iloc[:, :3].round(1).to_string())
print("elapsed %.0f s" % (time.time() - T0))

**Result (reading G2): partially reproduced.** Of 1745 medulla units in 19 sessions, under E1 with
the Bonferroni threshold 16.7% are selective in the sample epoch, 28.3% in the delay epoch and 63.4% in
the response epoch (paper 7, 21 and 49%); over the five seeds 16.6 to 17.6%, 27.7 to 28.7% and 63.4 to
63.9%. The order holds, and no value is within tolerance under E1, E2 or either threshold. By session
the sample value runs from 1.9 to 42.9%.


## Per-region subspaces, reading G2

Main reading and B2, the subspace generator seeded as in the main notebook. Then the variance
explained (M6, V1 to V3) and the paired t tests of fig4c, and the dimensionality of fig4e.

In [ ]:
def prepare(p, region_ids, short_runs_move):
    v = sessions[p]
    r = ms.rates([v["sess"]["spikes"][i] for i in region_ids], v["go"], W)
    z, nfb = ms.zscore(r, W)
    return z, ms.movement_mask(v["speed"], v["log_thr"], short_runs_move=short_runs_move), nfb


fits, fit_rows = {}, []
rng_sub = np.random.default_rng(seed("fig4c-alm-null-potent-similar"))
for p, s in alm_med.items():
    for reg in ("ALM", "medulla"):
        for rname in ("main", "B2"):
            z, mv, nfb = prepare(p, s[reg], rname == "B2")
            fit = ms.fit_region(z, mv, rng_sub)
            fit["proj"] = (z @ fit["qn"]).astype(np.float32)
            fits[(p, reg, rname)] = fit
            ve = ms.ve_readings(fit)
            fit_rows.append(dict(session=p.split("/")[1][:30], region=reg, reading=rname, units=z.shape[2],
                                 d_null=fit["d"][0], d_pot=fit["d"][1], d_null_raw=fit["d_raw"][0],
                                 moving=round(float((mv == 1).sum() / (mv >= 0).sum()), 2), fallback=nfb,
                                 z_absmax=round(float(np.abs(z).max())),
                                 **{f"{k} {n}": 100 * x for k, vv in ve.items() for n, x in zip(("null", "pot"), vv[:2])}))
    print(p.split("/")[1][:30], "done, elapsed %.0f s" % (time.time() - T0))
fit_df = pd.DataFrame(fit_rows)
print(fit_df.round(1).to_string())
print("peak memory %.0f MB" % (resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e3))

## Claims fig4c-alm-null-potent-similar and fig4c-medulla-null-smaller, reading G2

> In MAP ALM the movement-null and movement-potent subspaces explain comparable variance (36.0 +-
> 7.40% and 37.1 +- 8.70%; paired t test p = 0.15, 16 sessions).

> In the medullary reticular nuclei the movement-null subspace explains less variance than the
> movement-potent subspace (26.4 +- 3.90% against 44.8 +- 7.12%; p = 1.9e-9, 16 sessions).

In [ ]:
ve_rows = []
for (rname, reg), d in fit_df.groupby(["reading", "region"]):
    for V in ("V1", "V2", "V3"):
        a, b = d[f"{V} null"].values, d[f"{V} pot"].values
        t, pv = stats.ttest_rel(a, b)
        ve_rows.append(dict(region=reg, reading=rname, V=V, n=len(a), null_mean=a.mean(), null_sd=a.std(ddof=1),
                            pot_mean=b.mean(), pot_sd=b.std(ddof=1), diff=(a - b).mean(), t=t, p=pv,
                            null_below_pot=int((a < b).sum())))
ve_tab = pd.DataFrame(ve_rows)
print(ve_tab.round(3).to_string())
print("expected ALM null 36.0 +- 7.4, potent 37.1 +- 8.7 (p = 0.15); medulla null 26.4 +- 3.9, potent 44.8 +- 7.12 (p = 1.9e-9)")
alm_ok = ve_tab[(ve_tab.region == "ALM") & (abs(ve_tab.null_mean - 36.0) <= 7.4) & (abs(ve_tab.pot_mean - 37.1) <= 8.7)]
med_ok = ve_tab[(ve_tab.region == "medulla") & (abs(ve_tab.null_mean - 26.4) <= 3.9) & (abs(ve_tab.pot_mean - 44.8) <= 7.12)]
print("ALM readings with both means in tolerance:", alm_ok[["reading", "V", "null_mean", "pot_mean", "p"]].round(4).values.tolist())
print("medulla readings with both means in tolerance:", med_ok[["reading", "V", "null_mean", "pot_mean", "p"]].round(4).values.tolist())

**Result (reading G2).** fig4c-alm-null-potent-similar: under the main reading with V2, ALM null
and potent variance are 33.9 and 42.7%, both within tolerance of 36.0 and 37.1%, and the paired t
test finds no difference (p = 0.251 over 19 sessions; 0.249 to 0.261 over the seeds below), which the
criterion counts as reproduced. With V1 the main-reading p is 0.050 (0.050 to 0.058 over the seeds);
under B2 potent exceeds null (V1 p = 0.002, V2 p = 0.041) with means out of tolerance.
fig4c-medulla-null-smaller: null is below potent under the main reading (V1 46.3 against 67.3%,
p = 0.020; V2 35.7 against 51.7%, p = 0.027) and under B2 (V2 30.8 against 56.4%, p = 0.005), lower in
15 or 16 of 19 sessions, with the null mean outside 26.4 +- 3.9% in each, which the criterion counts as
partially reproduced. Single units dominate some regions: the largest |z| is above 1000 in 9 of the
38 regions, and in sub-460432_ses-20191218 the ALM null subspace holds 95.7% of the variance (V2).


Seed check: the main reading again under the claim's seed plus 1000 to 4000 (dimension shuffles and
the optimizer's start). B2 is run under the claim's seed only, to keep the notebook within the time
limit; in the main notebook B2 moved the t test p by less than 0.004 over the five seeds.

In [ ]:
seed_rows = []
for k in SEED_OFFSETS[1:]:
    rng_k = np.random.default_rng(seed("fig4c-alm-null-potent-similar", k))
    for p, s in alm_med.items():
        for reg in ("ALM", "medulla"):
            for rname in ("main",):
                z, mv, _ = prepare(p, s[reg], rname == "B2")
                fit = ms.fit_region(z, mv, rng_k)
                ve = ms.ve_readings(fit)
                seed_rows.append(dict(offset=k, reading=rname, session=p, region=reg, d_null_raw=fit["d_raw"][0],
                                      **{f"{V} {n}": 100 * x for V, vv in ve.items() for n, x in zip(("null", "pot"), vv[:2])}))
    print("offset", k, "elapsed %.0f s" % (time.time() - T0))
seed_df = pd.concat([pd.DataFrame(seed_rows),
                     fit_df.assign(offset=0, session=[next(p for p in alm_med if r in p) for r in fit_df.session])])
srows = []
for (off, rname, reg), d in seed_df.groupby(["offset", "reading", "region"]):
    for V in ("V1", "V2", "V3"):
        t, pv = stats.ttest_rel(d[f"{V} null"], d[f"{V} pot"])
        srows.append(dict(offset=off, reading=rname, region=reg, V=V, null=d[f"{V} null"].mean(), pot=d[f"{V} pot"].mean(), p=pv))
print(pd.DataFrame(srows).round(4).to_string())
for (off, rname), d in seed_df.groupby(["offset", "reading"]):
    d = d.sort_values("session")
    a, b = d[d.region == "ALM"].d_null_raw.values, d[d.region == "medulla"].d_null_raw.values
    print(f"offset {off} {rname}: N1 medulla {b.mean():.1f}, ALM {a.mean():.1f}, p = {stats.ttest_rel(b, a).pvalue:.3g}")
print("elapsed %.0f s" % (time.time() - T0))

## Claim fig4e-medulla-null-lower-dimension, reading G2

> The movement-null subspace has lower dimensionality in the medullary reticular nuclei than in ALM
> (paired t test, p = 7.4e-5).

N1 (shuffle dimension of the stationary activity) and N2 (participation ratio of the
movement-null covariance), main reading and B2.

In [ ]:
dim_rows = []
for p, s in alm_med.items():
    for rname in ("main", "B2"):
        row = dict(session=p.split("/")[1][:30], reading=rname)
        for reg in ("ALM", "medulla"):
            fit = fits[(p, reg, rname)]
            row[f"{reg} units"] = len(s[reg])
            row[f"{reg} N1"] = fit["d_raw"][0]
            row[f"{reg} N2"] = ms.participation_ratio(fit["qn"].T @ fit["C"] @ fit["qn"])
        dim_rows.append(row)
dim_df = pd.DataFrame(dim_rows)
print(dim_df.round(2).to_string())
for rname, d in dim_df.groupby("reading"):
    for m in ("N1", "N2"):
        t, pv = stats.ttest_rel(d[f"medulla {m}"], d[f"ALM {m}"])
        print(f"{rname} {m}: medulla mean {d[f'medulla {m}'].mean():.2f}, ALM mean {d[f'ALM {m}'].mean():.2f}, t = {t:.2f}, "
              f"p = {pv:.3g}, medulla lower in {(d[f'medulla {m}'] < d[f'ALM {m}']).sum()} of {len(d)}")

**Result (reading G2).** N1 does not differ (medulla 10.9, ALM 10.9, p = 0.98; 0.86 to 1.00 over
the seeds; under B2 p = 0.876). N2 is lower in the medulla (3.35 against 5.93, p = 0.027, lower in 14 of
19 sessions; under B2 3.72 against 6.92, p = 0.018), which the criterion counts as reproduced under N2.
In 5 sessions the medulla N2 is below 1.4, and each of these regions has a unit whose |z| exceeds 3000,
so N2 there measures one unit.


## Claim fig4d-decoding-asymmetry, reading G2

> Prediction is asymmetric; ALM movement-null activity is predicted less well from the medulla (21
> +- 6.7%) than the medulla from ALM (paired t test p = 1.2e-4, 16 sessions).

As in the main notebook (M8): ridge with lambda 0.01, readings T1 and T2, R1 and R2, predictors
standardized or not; main-reading subspaces; under the claim's seed and four more.

In [ ]:
bins = np.arange(int(round((ms.TASK_SPAN[0] - W[0]) / ms.BIN)), int(round((ms.TASK_SPAN[1] - W[0]) / ms.BIN)))
ALL_READ = [(s_, p_) for s_ in (False, True) for p_ in (True, False)]


def decode(rng, readings):
    out = []
    for p in alm_med:
        v = sessions[p]
        hit = np.array(v["sess"]["trials"]["outcome"])[v["keep"]] == "hit"
        for T in ("T1", "T2"):
            idx = np.arange(len(hit)) if T == "T1" else np.where(hit)[0]
            perm = rng.permutation(idx)
            ntr = int(round(0.7 * len(perm)))
            for src, tgt in (("ALM", "medulla"), ("medulla", "ALM")):
                xs, xt = fits[(p, src, "main")]["proj"], fits[(p, tgt, "main")]["proj"]
                for stdz, pooled in readings:
                    r2 = ms.ridge_r2(xs, xt, perm[:ntr], perm[ntr:], bins, stdz, pooled)
                    out.append(dict(session=p, T=T, src=src, std=stdz, R="R1" if pooled else "R2", r2=100 * r2.mean(),
                                    ppt=9 * xs.shape[2] / ntr))
    return pd.DataFrame(out)


def dec_summary(df):
    res = []
    for (T, stdz, R), d in df.groupby(["T", "std", "R"]):
        am_ = d[d.src == "ALM"].set_index("session").r2
        ma_ = d[d.src == "medulla"].set_index("session").r2
        res.append(dict(T=T, std=stdz, R=R, ALM_to_med=am_.mean(), med_to_ALM=ma_.mean(),
                        sem=ma_.std(ddof=1) / np.sqrt(len(ma_)), p_asym=stats.ttest_rel(am_, ma_.loc[am_.index]).pvalue,
                        sessions_above_0=int((ma_ > 0).sum())))
    return pd.DataFrame(res)


dec_runs = {k: decode(np.random.default_rng(seed("fig4d-alm-predicts-medulla-null", k)), ALL_READ if k == 0 else ALL_READ[::2])
            for k in SEED_OFFSETS}
print(dec_summary(dec_runs[0]).round(4).to_string())
print("expected medulla to ALM 21 +- 6.7, below ALM to medulla (p = 1.2e-4)")
for k in SEED_OFFSETS[1:]:
    print("offset", k)
    print(dec_summary(dec_runs[k]).round(4).to_string())
d0 = dec_runs[0][(dec_runs[0]["T"] == "T1") & (~dec_runs[0]["std"]) & (dec_runs[0].R == "R1")]
print("T1, unstandardized, R1, per session: R2 above 0 in %d of %d; Spearman correlation of R2 with predictors per training trial %.2f"
      % ((d0.r2 > 0).sum(), len(d0), stats.spearmanr(d0.r2, d0.ppt).correlation))
print("elapsed %.0f s, peak memory %.0f MB" % (time.time() - T0, resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e3))

**Result (reading G2): not reproduced.** Every mean R2 is below -10^6 % (no session has a medulla to
ALM R2 above zero), and the paired t test of ALM to medulla against medulla to ALM gives p = 0.15 to 0.46
under the claim's seed and 0.17 to 0.87 over the other seeds.

## Summary (reading G2)

| Claim | Under G2 |
|---|---|
| fig4b-medulla-selectivity | Partially reproduced (16.7, 28.3, 63.4%) |
| fig4c-alm-null-potent-similar | Reproduced (main reading, V2: 33.9 against 42.7%, p = 0.251) |
| fig4c-medulla-null-smaller | Partially reproduced (test holds, null mean out of tolerance) |
| fig4e-medulla-null-lower-dimension | Reproduced under N2 (p = 0.027), not under N1 (p = 0.98) |
| fig4d-decoding-asymmetry | Not reproduced |

The attempt's outcomes rest on reading G1 (main notebook), whose bilateral unit counts are near the
paper's. G1 is held fixed while the readings of M3 to M6 are searched, so the two claims that hold here
under G2 are recorded as not reproduced or partially reproduced under G1; G2 gives about 2.5 times the paper's units and transfers CCF labels from the nearest unit
classified good. A failure to reproduce here may be a failure of this attempt, not of the paper: see
the reasoning and the differences from the paper's method.

The shuffle dimensions of a few regions here, and with them the variance explained, the
dimensionality and the decoding R2, differ in their last digits or more between processors (see the
main notebook's summary); an earlier execution on another runner gave the same verdicts.
